In [1]:
# --- 1. Импорты и общие настройки ---
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style("whitegrid")
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from pmdarima import auto_arima
from statsmodels.tsa.statespace.sarimax import SARIMAX
from prophet import Prophet  # pip install prophet
from sklearn.metrics import mean_squared_error, mean_absolute_error
import warnings
pd.options.display.float_format = '{:.2f}'.format
warnings.filterwarnings("ignore")
# plt.style.use("seaborn-whitegrid")



In [2]:
# --- 2. Загрузка данных ---
# df должен содержать столбцы: "Регион", "Период" (YYYY-MM), и целевой столбец с показателем.
df = pd.read_excel("Датасет по яйцам v2.xlsx")
df["Период"] = pd.to_datetime(df["Период"], format="%Y-%m")
df.sample(10)



,Регион,Период,Яйца
48,АКМОЛИНСКАЯ ОБЛАСТЬ,2019-01-01,72826.40
636,ГАЛМАТЫ,2015-02-01,35.50
1469,КОСТАНАЙСКАЯ ОБЛАСТЬ,2024-08-01,34474.80
1919,ПАВЛОДАРСКАЯ ОБЛАСТЬ,2020-11-01,20677.80
367,АЛМАТИНСКАЯ ОБЛАСТЬ,2024-06-01,44604.40
2090,СЕВЕРО-КАЗАХСТАНСКАЯ ОБЛАСТЬ,2024-07-01,58261.10
1337,КАРАГАНДИНСКАЯ ОБЛАСТЬ,2024-03-01,56942.50
205,АКТЮБИНСКАЯ ОБЛАСТЬ,2021-07-01,20898.00
350,АЛМАТИНСКАЯ ОБЛАСТЬ,2023-01-01,40590.30
2076,СЕВЕРО-КАЗАХСТАНСКАЯ ОБЛАСТЬ,2023-05-01,51580.20


In [3]:
regions = df['Регион'].unique()
target   = "Яйца"
horizon  = 3
epsilon = 1e-6

In [4]:
first_test = pd.to_datetime("2024-08-01")
last_possible = df["Период"].max() - pd.DateOffset(months=horizon-1)
test_starts = pd.date_range(first_test, last_possible, freq="MS")

## Holt-Winter's (log)

In [5]:
results_hw = []
for region in regions:
    ts = (df[df["Регион"] == region]
          .set_index("Период")[target]
          .dropna()
          .sort_index())
    if len(ts) < 24:
        print(f"{region}: всего {len(ts)} мес. — сезонный Holt-Winter's невозможен.")
        continue

    for test_start in test_starts:
        test_end = test_start + pd.DateOffset(months=horizon) - pd.DateOffset(days=1)

        # формируем train / test
        train = ts[ts.index < test_start]
        test  = ts[(ts.index >= test_start) & (ts.index <= test_end)]

        # пропускаем, если недостаточно данных или неполный test
        if len(train) < 24 or len(test) < horizon:
            continue

        # обучаем модель
        train_log = np.log1p(train)

        hw_log = ExponentialSmoothing(
            train_log,
            seasonal="add",
            seasonal_periods=12
        ).fit(optimized=True)

        # прогноз и метрики
        fc_log = hw_log.forecast(horizon)
        fc = np.expm1(fc_log) 
        # fc   = model.forecast(horizon)
        rmse = np.sqrt(mean_squared_error(test, fc))
        mae  = mean_absolute_error(test, fc)
        mape = (np.abs((test - fc) / test).mean()) * 100

        results_hw.append({
            "Регион":      region,
            "Test start":  test_start.strftime("%Y-%m"),
            "Test end":    test_end.strftime("%Y-%m"),
            "Forecast":    [x.round(2) for x in list(fc.values)],
            "Actual":      [y.round(2) for y in list(test.values)],
            "RMSE":        rmse,
            "MAE":         mae,
            "MAPE_%":      mape
        })

# 4) Усреднение по всем скользящим окнам для каждого региона
res_hw = pd.DataFrame(results_hw)
res_hw.to_excel("results/Яйца - Результаты прогнозов ХВ v2.xlsx", index=False)
print("Результаты прогнозов HW на 3 месяца:")

display(res_hw)

final_hw = (
    res_hw
    .groupby("Регион")[["RMSE","MAE","MAPE_%"]]
    .mean()
    .round(2)
    .reset_index()
)
final_hw.to_excel("results/Яйца - Результаты прогнозов ХВ средние v2.xlsx", index=False)
print("Средние метрики Holt–Winter's по регионам (rolling-3):")
display(final_hw)

Результаты прогнозов HW на 3 месяца:


,Регион,Test start,Test end,Forecast,Actual,RMSE,MAE,MAPE_%
0,АКМОЛИНСКАЯ ОБЛАСТЬ,2024-08,2024-10,"[59945.34, 57559.29, 56489.42]","[62899.9, 58300.1, 56071.5]",1775.10,1371.10,2.24
1,АКМОЛИНСКАЯ ОБЛАСТЬ,2024-09,2024-11,"[59058.65, 57947.67, 54167.51]","[58300.1, 56071.5, 52213.7]",1624.07,1529.51,2.80
2,АКМОЛИНСКАЯ ОБЛАСТЬ,2024-10,2024-12,"[57548.43, 53798.82, 58236.09]","[56071.5, 52213.7, 55640.3]",1952.10,1885.95,3.45
3,АКМОЛИНСКАЯ ОБЛАСТЬ,2024-11,2025-01,"[53057.6, 57451.73, 53151.03]","[52213.7, 55640.3, 54329.0]",1339.28,1277.77,2.35
4,АКМОЛИНСКАЯ ОБЛАСТЬ,2024-12,2025-02,"[56953.5, 52698.63, 51168.69]","[55640.3, 54329.0, 45131.6]",3689.13,2993.55,6.25
...,...,...,...,...,...,...,...,...
195,ТУРКЕСТАНСКАЯ ОБЛАСТЬ,2025-01,2025-03,"[17490.44, 17231.92, 23998.96]","[18830.5, 16545.6, 21272.4]",1798.23,1584.31,8.03
196,ТУРКЕСТАНСКАЯ ОБЛАСТЬ,2025-02,2025-04,"[18534.04, 25788.23, 27580.57]","[16545.6, 21272.4, 23757.1]",3603.96,3442.58,16.45
197,ТУРКЕСТАНСКАЯ ОБЛАСТЬ,2025-03,2025-05,"[23512.96, 25178.19, 26315.98]","[21272.4, 23757.1, 25319.1]",1636.40,1552.84,6.82
198,ТУРКЕСТАНСКАЯ ОБЛАСТЬ,2025-04,2025-06,"[23065.8, 24137.61, 22551.23]","[23757.1, 25319.1, 27352.3]",2882.37,2224.62,8.38


Средние метрики Holt–Winter's по регионам (rolling-3):


,Регион,RMSE,MAE,MAPE_%
0,АКМОЛИНСКАЯ ОБЛАСТЬ,2688.42,2369.14,4.42
1,АКТЮБИНСКАЯ ОБЛАСТЬ,1374.14,1272.95,6.64
2,АЛМАТИНСКАЯ ОБЛАСТЬ,4632.18,4041.75,9.16
3,АТЫРАУСКАЯ ОБЛАСТЬ,536.35,503.07,12.95
4,ВОСТОЧНО-КАЗАХСТАНСКАЯ ОБЛАСТЬ,340.97,301.33,8.49
5,ГАЛМАТЫ,1.89,1.71,11.53
6,ГАСТАНА,0.22,0.20,NaN
7,ГШЫМКЕНТ,1959.45,1695.09,9.08
8,ЖАМБЫЛСКАЯ ОБЛАСТЬ,1153.00,1059.44,13.57
9,ЗАПАДНО-КАЗАХСТАНСКАЯ ОБЛАСТЬ,946.96,821.46,7.66


## SARIMA

In [6]:
results_sarima = []

for region in regions:
    ts = (
        df[df["Регион"] == region]
        .set_index("Период")[target]
        .dropna()
        .sort_index()
    )
    ts = ts + epsilon
    ts_log = np.log(ts)

    if len(ts_log) < 12 + horizon:
        print(f"{region}: менее {12+horizon} мес. для авто-ARIMA, пропускаем.")
        continue

    for test_start in test_starts:
        test_end = test_start + pd.DateOffset(months=horizon) - pd.DateOffset(days=1)

        train_log = ts_log[ts_log.index < test_start]
        test_log  = ts_log[(ts_log.index >= test_start) & (ts_log.index <= test_end)]
        if len(train_log) < 12 + horizon or len(test_log) < horizon:
            continue

        # автоподбор на лог-данных
        use_seasonal = len(train_log) >= 2 * 12

        sarima_log = auto_arima(
            train_log,
            seasonal=use_seasonal,
            m=12 if use_seasonal else 1,
            D=1 if use_seasonal else 0,      # фиксируем порядок сезонной разности
            seasonal_test=None,               # пропустить nsdiffs
            boxcox=True,
            stepwise=True,
            suppress_warnings=True,
            error_action="ignore"
        )
      
        # прогноз в лог-шкале
        fc_log = sarima_log.predict(n_periods=horizon, return_conf_int=False)

        # возвращаем прогноз в исходные единицы
        fc = np.exp(fc_log) - epsilon
        actual = np.exp(test_log.values) - epsilon  # но exp(log(x)) == x

        # метрики на исходном уровне
        rmse = np.sqrt(mean_squared_error(actual, fc))
        mae  = mean_absolute_error(actual, fc)
        mape = (np.abs((actual - fc) / actual).mean()) * 100

        results_sarima.append({
            "Регион":         region,
            "Test start":     test_start.strftime("%Y-%m"),
            "Test end":       test_end.strftime("%Y-%m"),
            "order":          sarima_log.order,
            "seasonal_order": sarima_log.seasonal_order,
            "RMSE":           round(rmse,2),
            "MAE":            round(mae,2),
            "MAPE_%":         round(mape,2),
            "Forecast":       [round(x,2) for x in fc],
            "Actual":         [round(y,2) for y in actual]
        })
# формируем DataFrame с результатами
res_sarima = pd.DataFrame(results_sarima)
res_sarima.to_excel("results/Яйца - Результаты прогнозов SARIMA v2.xlsx", index=False)
print("Результаты прогнозов SARIMA на 3 месяца:")

display(res_sarima)

final_sarima = (
    res_sarima
    .groupby("Регион")[["RMSE","MAE","MAPE_%"]]
    .mean()
    .round(2)
    .reset_index()
)
final_sarima.to_excel("results/Яйца - Результаты прогнозов SARIMA средние v2.xlsx", index=False)
print("Средние метрики SARIMA по регионам (rolling-3):")
display(final_sarima)

Результаты прогнозов SARIMA на 3 месяца:


,Регион,Test start,Test end,order,seasonal_order,RMSE,MAE,MAPE_%,Forecast,Actual
0,АКМОЛИНСКАЯ ОБЛАСТЬ,2024-08,2024-10,"(3, 0, 0)","(0, 1, 1, 12)",1824.86,1669.05,2.79,"[60272.7, 57467.94, 57619.28]","[62899.9, 58300.1, 56071.5]"
1,АКМОЛИНСКАЯ ОБЛАСТЬ,2024-09,2024-11,"(3, 0, 0)","(0, 1, 1, 12)",2188.60,1971.60,3.62,"[58928.43, 58743.29, 54828.39]","[58300.1, 56071.5, 52213.7]"
2,АКМОЛИНСКАЯ ОБЛАСТЬ,2024-10,2024-12,"(1, 0, 1)","(0, 1, 1, 12)",2553.07,2492.56,4.55,"[58343.66, 54167.16, 58892.35]","[56071.5, 52213.7, 55640.3]"
3,АКМОЛИНСКАЯ ОБЛАСТЬ,2024-11,2025-01,"(2, 0, 1)","(0, 1, 1, 12)",1415.38,1322.64,2.43,"[53039.24, 57653.7, 53200.01]","[52213.7, 55640.3, 54329.0]"
4,АКМОЛИНСКАЯ ОБЛАСТЬ,2024-12,2025-02,"(3, 0, 0)","(0, 1, 1, 12)",4165.61,3318.96,6.94,"[57444.28, 53042.49, 51998.0]","[55640.3, 54329.0, 45131.6]"
...,...,...,...,...,...,...,...,...,...,...
195,ТУРКЕСТАНСКАЯ ОБЛАСТЬ,2025-01,2025-03,"(1, 0, 1)","(0, 1, 1, 12)",1934.51,1552.63,7.68,"[17097.09, 16487.99, 24139.27]","[18830.5, 16545.6, 21272.4]"
196,ТУРКЕСТАНСКАЯ ОБЛАСТЬ,2025-02,2025-04,"(1, 0, 1)","(0, 1, 1, 12)",2765.48,2636.23,12.62,"[18082.67, 24833.94, 26567.17]","[16545.6, 21272.4, 23757.1]"
197,ТУРКЕСТАНСКАЯ ОБЛАСТЬ,2025-03,2025-05,"(1, 0, 1)","(0, 1, 1, 12)",1959.52,1886.54,8.21,"[23497.9, 26052.84, 26457.49]","[21272.4, 23757.1, 25319.1]"
198,ТУРКЕСТАНСКАЯ ОБЛАСТЬ,2025-04,2025-06,"(1, 0, 1)","(0, 1, 1, 12)",1685.97,1333.14,5.03,"[24156.59, 26147.42, 24580.7]","[23757.1, 25319.1, 27352.3]"


Средние метрики SARIMA по регионам (rolling-3):


,Регион,RMSE,MAE,MAPE_%
0,АКМОЛИНСКАЯ ОБЛАСТЬ,2894.71,2489.20,4.70
1,АКТЮБИНСКАЯ ОБЛАСТЬ,1580.75,1465.02,7.65
2,АЛМАТИНСКАЯ ОБЛАСТЬ,4030.44,3685.44,8.23
3,АТЫРАУСКАЯ ОБЛАСТЬ,816.00,752.81,19.69
4,ВОСТОЧНО-КАЗАХСТАНСКАЯ ОБЛАСТЬ,359.73,313.61,8.02
5,ГАЛМАТЫ,0.94,0.94,6.30
6,ГАСТАНА,0.07,0.06,35.48
7,ГШЫМКЕНТ,2834.03,2546.49,13.82
8,ЖАМБЫЛСКАЯ ОБЛАСТЬ,1460.24,1383.31,20.33
9,ЗАПАДНО-КАЗАХСТАНСКАЯ ОБЛАСТЬ,864.00,746.92,6.94


## Facebook Prophet

In [7]:
results_prophet = []

for region in regions:
    ts = (
        df[df["Регион"] == region]
        .set_index("Период")[target]
        .dropna()
        .sort_index()
    )
    if len(ts) < 12 + horizon:
        print(f"{region}: менее {12+horizon} мес. данных, пропускаем.")
        continue

    for test_start in test_starts:
        test_end = test_start + pd.DateOffset(months=horizon) - pd.DateOffset(days=1)

        train = ts[ts.index < test_start]
        test  = ts[(ts.index >= test_start) & (ts.index <= test_end)]
        if len(train) < 12 + horizon or len(test) < horizon:
            continue

        # Подготовка данных для Prophet
#         df_prophet = train.reset_index().rename(columns={"Период":"ds", target:"y"})
# # подготовка для одного региона
        df_prophet = train.reset_index().rename(columns={"Период":"ds", target:"y"})
        df_prophet["y"] = np.log(df_prophet["y"] + epsilon)

        m = Prophet()
        m.fit(df_prophet)

        future = m.make_future_dataframe(periods=horizon, freq="MS")
        forecast = m.predict(future)

        # берем только прогнозные точки
        yhat_log = forecast["yhat"].values[-horizon:]
        fc = np.exp(yhat_log) - epsilon

        # m = Prophet()
        # m.fit(df_prophet)

        # # Создаем DataFrame будущих дат и делаем прогноз
        # # future = m.make_future_dataframe(periods=horizon, freq="MS")
        # # forecast = m.predict(future)

        # # Отбираем только наши горизонты
        # fc = forecast.set_index("ds")["yhat"].loc[test.index].values
        actual = test.values

        # Расчет метрик
        rmse  = np.sqrt(mean_squared_error(actual, fc))
        mae   = mean_absolute_error(actual, fc)
        mape  = (np.abs((actual - fc) / actual).mean()) * 100

        results_prophet.append({
            "Регион":     region,
            "Test start": test_start.strftime("%Y-%m"),
            "Test end":   test_end.strftime("%Y-%m"),
            "RMSE":       round(rmse, 2),
            "MAE":        round(mae, 2),
            "MAPE_%":     round(mape, 2),
            "Forecast":   [round(x, 2) for x in fc],
            "Actual":     [round(x, 2) for x in actual]
        })

# Собираем результаты в DataFrame
res_prophet = pd.DataFrame(results_prophet)
res_prophet.to_excel("results/Яйца - Результаты прогнозов Prophet v2.xlsx", index=False)
print("Результаты прогнозов Prophet на 3 месяца:")
display(res_prophet)

final_prophet = (
    res_prophet
    .groupby("Регион")[["RMSE","MAE","MAPE_%"]]
    .mean()
    .round(2)
    .reset_index()
)
final_prophet.to_excel("results/Яйца - Результаты прогнозов Prophet средние v2.xlsx", index=False)
print("Средние метрики Prophet по регионам (rolling-3):")
display(final_prophet)


14:26:02 - cmdstanpy - INFO - Chain [1] start processing
14:26:02 - cmdstanpy - INFO - Chain [1] done processing
14:26:03 - cmdstanpy - INFO - Chain [1] start processing
14:26:03 - cmdstanpy - INFO - Chain [1] done processing
14:26:03 - cmdstanpy - INFO - Chain [1] start processing
14:26:03 - cmdstanpy - INFO - Chain [1] done processing
14:26:03 - cmdstanpy - INFO - Chain [1] start processing
14:26:03 - cmdstanpy - INFO - Chain [1] done processing
14:26:04 - cmdstanpy - INFO - Chain [1] start processing
14:26:04 - cmdstanpy - INFO - Chain [1] done processing
14:26:04 - cmdstanpy - INFO - Chain [1] start processing
14:26:04 - cmdstanpy - INFO - Chain [1] done processing
14:26:04 - cmdstanpy - INFO - Chain [1] start processing
14:26:04 - cmdstanpy - INFO - Chain [1] done processing
14:26:04 - cmdstanpy - INFO - Chain [1] start processing
14:26:04 - cmdstanpy - INFO - Chain [1] done processing
14:26:05 - cmdstanpy - INFO - Chain [1] start processing
14:26:05 - cmdstanpy - INFO - Chain [1]

Результаты прогнозов Prophet на 3 месяца:


,Регион,Test start,Test end,RMSE,MAE,MAPE_%,Forecast,Actual
0,АКМОЛИНСКАЯ ОБЛАСТЬ,2024-08,2024-10,4704.29,4352.76,7.31,"[56567.11, 56292.1, 51354.0]","[62899.9, 58300.1, 56071.5]"
1,АКМОЛИНСКАЯ ОБЛАСТЬ,2024-09,2024-11,2635.58,2252.69,4.03,"[56815.63, 51896.9, 51114.68]","[58300.1, 56071.5, 52213.7]"
2,АКМОЛИНСКАЯ ОБЛАСТЬ,2024-10,2024-12,2925.20,2611.96,4.71,"[51869.76, 51236.92, 52982.94]","[56071.5, 52213.7, 55640.3]"
3,АКМОЛИНСКАЯ ОБЛАСТЬ,2024-11,2025-01,2239.01,2045.56,3.75,"[51399.66, 53304.2, 51342.46]","[52213.7, 55640.3, 54329.0]"
4,АКМОЛИНСКАЯ ОБЛАСТЬ,2024-12,2025-02,2024.74,1955.74,3.73,"[53637.78, 51756.07, 46423.36]","[55640.3, 54329.0, 45131.6]"
...,...,...,...,...,...,...,...,...
195,ТУРКЕСТАНСКАЯ ОБЛАСТЬ,2025-01,2025-03,2116.19,2108.54,11.34,"[16931.27, 18883.66, 23360.73]","[18830.5, 16545.6, 21272.4]"
196,ТУРКЕСТАНСКАЯ ОБЛАСТЬ,2025-02,2025-04,2507.34,2504.18,12.57,"[19193.05, 23796.07, 26098.53]","[16545.6, 21272.4, 23757.1]"
197,ТУРКЕСТАНСКАЯ ОБЛАСТЬ,2025-03,2025-05,1677.44,1650.60,7.16,"[23264.13, 25453.3, 26582.97]","[21272.4, 23757.1, 25319.1]"
198,ТУРКЕСТАНСКАЯ ОБЛАСТЬ,2025-04,2025-06,1793.09,1642.40,6.35,"[25137.02, 26241.31, 24727.25]","[23757.1, 25319.1, 27352.3]"


Средние метрики Prophet по регионам (rolling-3):


,Регион,RMSE,MAE,MAPE_%
0,АКМОЛИНСКАЯ ОБЛАСТЬ,2756.28,2520.45,4.44
1,АКТЮБИНСКАЯ ОБЛАСТЬ,2416.51,2292.86,12.02
2,АЛМАТИНСКАЯ ОБЛАСТЬ,6669.50,6234.80,13.93
3,АТЫРАУСКАЯ ОБЛАСТЬ,3201.92,3188.79,80.04
4,ВОСТОЧНО-КАЗАХСТАНСКАЯ ОБЛАСТЬ,536.02,493.53,11.21
5,ГАЛМАТЫ,8.09,5.66,38.15
6,ГАСТАНА,0.11,0.10,63.37
7,ГШЫМКЕНТ,1942.62,1798.74,9.49
8,ЖАМБЫЛСКАЯ ОБЛАСТЬ,1535.33,1487.32,24.05
9,ЗАПАДНО-КАЗАХСТАНСКАЯ ОБЛАСТЬ,953.55,838.84,7.50


In [8]:
# Переименуем колонки с MAPE, чтобы было понятно, к какому методу относятся
hw = final_hw.rename(columns={"MAPE_%": "MAPE_HW"})
sar = final_sarima.rename(columns={"MAPE_%": "MAPE_SARIMA"})
pr  = final_prophet.rename(columns={"MAPE_%": "MAPE_Prophet"})

# Мёрджим по региону
summary = (
    hw[["Регион", "MAPE_HW"]]
    .merge(sar[["Регион", "MAPE_SARIMA"]], on="Регион")
    .merge(pr[["Регион", "MAPE_Prophet"]], on="Регион")
)

# Определяем для каждой строки, какой столбец MAPE минимален
# idxmin вернёт название столбца с минимальным значением
summary["Best_method"] = summary[["MAPE_HW","MAPE_SARIMA","MAPE_Prophet"]] \
                           .idxmin(axis=1) \
                           .str.replace("MAPE_","")  # убираем префикс для красоты

# Если нужно, можно сразу отсортировать
# summary = summary.sort_values("Best_method")

# допустим, у вас уже есть summary
summary = summary.round({
    "MAPE_HW": 2,
    "MAPE_SARIMA": 2,
    "MAPE_Prophet": 2
})

# Готово!
print(summary.to_string(index=False))
summary.to_excel("results/Яйца - Лучшие модели v2.xlsx", index=False)


                        Регион  MAPE_HW  MAPE_SARIMA  MAPE_Prophet Best_method
           АКМОЛИНСКАЯ ОБЛАСТЬ     4.42         4.70          4.44          HW
           АКТЮБИНСКАЯ ОБЛАСТЬ     6.64         7.65         12.02          HW
           АЛМАТИНСКАЯ ОБЛАСТЬ     9.16         8.23         13.93      SARIMA
            АТЫРАУСКАЯ ОБЛАСТЬ    12.95        19.69         80.04          HW
ВОСТОЧНО-КАЗАХСТАНСКАЯ ОБЛАСТЬ     8.49         8.02         11.21      SARIMA
                       ГАЛМАТЫ    11.53         6.30         38.15      SARIMA
                       ГАСТАНА      NaN        35.48         63.37      SARIMA
                      ГШЫМКЕНТ     9.08        13.82          9.49          HW
            ЖАМБЫЛСКАЯ ОБЛАСТЬ    13.57        20.33         24.05          HW
 ЗАПАДНО-КАЗАХСТАНСКАЯ ОБЛАСТЬ     7.66         6.94          7.50      SARIMA
        КАРАГАНДИНСКАЯ ОБЛАСТЬ     6.58         6.68          8.83          HW
          КОСТАНАЙСКАЯ ОБЛАСТЬ    17.75        15.62

In [9]:
#FOLDER = Path("results")  # папка, где лежат файлы
FILE_HW      = "results/Яйца - Результаты прогнозов ХВ средние v2.xlsx"
FILE_SARIMA  = "results/Яйца - Результаты прогнозов SARIMA средние v2.xlsx"
FILE_PROPHET = "results/Яйца - Результаты прогнозов Prophet средние v2.xlsx"

# === Загрузка исходных таблиц ===
final_hw      = pd.read_excel(FILE_HW)
final_sarima  = pd.read_excel(FILE_SARIMA)
final_prophet = pd.read_excel(FILE_PROPHET)

# Ожидаемые столбцы: 'Регион', 'MAPE_%', 'MAE' (и/или 'RMSE')
# Переименуем для прозрачности
hw = final_hw.rename(columns={"MAPE_%": "MAPE_HW", "MAE": "MAE_HW"})[["Регион","MAPE_HW","MAE_HW"]]
sar = final_sarima.rename(columns={"MAPE_%": "MAPE_SARIMA", "MAE": "MAE_SARIMA"})[["Регион","MAPE_SARIMA","MAE_SARIMA"]]
pr  = final_prophet.rename(columns={"MAPE_%": "MAPE_Prophet", "MAE": "MAE_Prophet"})[["Регион","MAPE_Prophet","MAE_Prophet"]]

# === Объединяем по региону ===
summary = (
    hw.merge(sar, on="Регион", how="inner")
      .merge(pr,  on="Регион", how="inner")
)


In [10]:
THRESHOLD_MAPE = 1000.0  # порог

mape_cols = ["MAPE_HW","MAPE_SARIMA","MAPE_Prophet"]
mae_cols  = ["MAE_HW","MAE_SARIMA","MAE_Prophet"]

# 1) Приведём метрики к числам (на всякий случай ещё раз)
for c in mape_cols + mae_cols:
    summary[c] = pd.to_numeric(summary[c], errors="coerce")

def choose_best_simple(row):
    # Берём числовые серии и подменяем NaN на +inf, чтобы .idxmin() стабильно работал
    mape_s = row[mape_cols].astype(float).fillna(np.inf)
    mae_s  = row[mae_cols].astype(float).fillna(np.inf)

    min_mape = mape_s.min()

    # Если все MAPE были NaN -> min = +inf
    if np.isinf(min_mape):
        criterion = "MAE"
        winner_col = mae_s.idxmin()
    elif min_mape <= THRESHOLD_MAPE:
        criterion = "MAPE"
        winner_col = mape_s.idxmin()
    else:
        criterion = "MAE"
        winner_col = mae_s.idxmin()

    method = winner_col.split("_")[-1]  # HW / SARIMA / Prophet

    return pd.Series({
        "Best_method": method,
        "Best_criterion": criterion,
        "Best_MAPE": float(mape_s.replace(np.inf, np.nan).min()),
        "Best_MAE": float(mae_s.replace(np.inf, np.nan).min())
    })

best = summary.apply(choose_best_simple, axis=1)

result = pd.concat([summary, best], axis=1)

# Округление и сохранение
for c in mape_cols + mae_cols + ["Best_MAPE","Best_MAE"]:
    result[c] = result[c].round(2)

# Если у вас есть переменная OUT_FILE — используйте её. Иначе:
OUT_FILE = "results/Яйца - Лучшие модели (MAPE_then_MAE) v2.xlsx"
result.sort_values(["Best_method","Регион"]).to_excel(OUT_FILE, index=False)

result

,Регион,MAPE_HW,MAE_HW,MAPE_SARIMA,MAE_SARIMA,MAPE_Prophet,MAE_Prophet,Best_method,Best_criterion,Best_MAPE,Best_MAE
0,АКМОЛИНСКАЯ ОБЛАСТЬ,4.42,2369.14,4.70,2489.20,4.44,2520.45,HW,MAPE,4.42,2369.14
1,АКТЮБИНСКАЯ ОБЛАСТЬ,6.64,1272.95,7.65,1465.02,12.02,2292.86,HW,MAPE,6.64,1272.95
2,АЛМАТИНСКАЯ ОБЛАСТЬ,9.16,4041.75,8.23,3685.44,13.93,6234.80,SARIMA,MAPE,8.23,3685.44
3,АТЫРАУСКАЯ ОБЛАСТЬ,12.95,503.07,19.69,752.81,80.04,3188.79,HW,MAPE,12.95,503.07
4,ВОСТОЧНО-КАЗАХСТАНСКАЯ ОБЛАСТЬ,8.49,301.33,8.02,313.61,11.21,493.53,SARIMA,MAPE,8.02,301.33
5,ГАЛМАТЫ,11.53,1.71,6.30,0.94,38.15,5.66,SARIMA,MAPE,6.30,0.94
6,ГАСТАНА,NaN,0.20,35.48,0.06,63.37,0.10,SARIMA,MAPE,35.48,0.06
7,ГШЫМКЕНТ,9.08,1695.09,13.82,2546.49,9.49,1798.74,HW,MAPE,9.08,1695.09
8,ЖАМБЫЛСКАЯ ОБЛАСТЬ,13.57,1059.44,20.33,1383.31,24.05,1487.32,HW,MAPE,13.57,1059.44
9,ЗАПАДНО-КАЗАХСТАНСКАЯ ОБЛАСТЬ,7.66,821.46,6.94,746.92,7.50,838.84,SARIMA,MAPE,6.94,746.92
